# NBA Golden Dataset — Upload

This notebook creates the offline evaluation dataset for the multi-turn NBA credit-card chatbot.

Each example is a **scripted conversation** — a list of user turns the evaluator will replay against the graph one turn at a time. The reference output records which offer we expect the bot to end up recommending (or that we expect the **customer-risk guardrail** to fire and decline the conversation).

Runs against LangSmith via `client.create_dataset` + `client.create_examples`, mirroring the pattern in `dataset_upload.ipynb` from the LangSmith course.

> **Note on risk-decline examples.** The guardrail (rules-based, in `app/nba_app.py`) scores off `state["customer_record"]` loaded from the SQLite `customers` table — not off the user's utterances. For a `risk-decline` example to actually trigger the guardrail, the `customer_id` needs to point at a row with `risk_tier == 'HIGH'`, high debt-to-income, or unemployment with material debt. The IDs used below (**901, 902**) are chosen from the HIGH-risk end of the seeded population; adjust if your `nba_demo.db` was seeded with a different `--rows` / seed.

## Setup

In [ ]:
import os
# You can set these inline or via a .env file.
# os.environ['LANGSMITH_API_KEY'] = '...'
os.environ.setdefault('LANGSMITH_TRACING', 'true')
os.environ.setdefault('LANGSMITH_PROJECT', 'nba-demo')

from dotenv import load_dotenv
load_dotenv(override=True)

In [ ]:
from langsmith import Client

client = Client()
DATASET_NAME = 'NBA Golden Dataset'

## Define scripted conversations

Each example is:

```python
inputs  = {'turns': [<user turn 1>, <user turn 2>, ...], 'customer_id': <int>}
outputs = {'expected_offer_id': 'PLAT_TRAVEL', 'expected_path': 'proceed', 'min_turns_to_offer': 2, 'max_turns_to_offer': 4}
```

Splits: `travel`, `balance-transfer`, `secured`, `student`, `cashback`, `risk-decline`. The `split` label lets us slice experiments by offer type from `nba_experiments.ipynb`. The `risk-decline` split covers the customer-risk guardrail path — those examples describe customers with HIGH-risk profiles (heavy debt-to-income, unemployment, poor payment history) so the bot's decline is grounded in what the customer said, even though the guardrail itself keys off the DB row.

In [ ]:
EXAMPLES = [
    # ---- travel ----
    dict(split='travel', customer_id=101,
         turns=[
            "Hi, I'm thinking about upgrading my credit card.",
            "I travel a lot for work — I fly maybe twice a month. I earn about $180,000 a year and I'm employed full-time.",
         ], expected_offer_id='PLAT_TRAVEL', expected_path='proceed'),
    dict(split='travel', customer_id=102,
         turns=[
            "Looking for a premium card.",
            "I'd love lounge access and points on hotels. Annual income is around $150K.",
         ], expected_offer_id='PLAT_TRAVEL', expected_path='proceed'),
    dict(split='travel', customer_id=103,
         turns=[
            "I want to earn miles.",
            "I'm 34, employed, make $120K, and I spend a lot on flights and restaurants.",
         ], expected_offer_id='PLAT_TRAVEL', expected_path='proceed'),

    # ---- balance-transfer ----
    dict(split='balance-transfer', customer_id=201,
         turns=[
            "I'm carrying some credit card debt and want to consolidate.",
            "I owe about $8,000 across two cards. I make $55K a year, employed.",
         ], expected_offer_id='BALANCE_TRANSFER', expected_path='proceed'),
    dict(split='balance-transfer', customer_id=202,
         turns=[
            "Any card that can help me pay less interest?",
            "I've got roughly $12,000 in existing debt, income around $60K.",
         ], expected_offer_id='BALANCE_TRANSFER', expected_path='proceed'),
    dict(split='balance-transfer', customer_id=203,
         turns=[
            "Need to consolidate a couple of balances.",
            "I'm 45, employed, earning about $70K, and I have $10K existing debt.",
         ], expected_offer_id='BALANCE_TRANSFER', expected_path='proceed'),

    # ---- secured builder ----
    dict(split='secured', customer_id=301,
         turns=[
            "I want to rebuild my credit.",
            "I had some late payments last year. I'm employed and make around $30K.",
         ], expected_offer_id='SECURED_BUILDER', expected_path='proceed'),
    dict(split='secured', customer_id=302,
         turns=[
            "My credit score is pretty low, I need help.",
            "I'd put down a deposit if it helps. Income is about $25K.",
         ], expected_offer_id='SECURED_BUILDER', expected_path='proceed'),

    # ---- student ----
    dict(split='student', customer_id=401,
         turns=[
            "I'm a college student and I want to start building credit.",
            "I'm 20, in school, and I work part-time earning maybe $8K a year.",
         ], expected_offer_id='STUDENT_STARTER', expected_path='proceed'),
    dict(split='student', customer_id=402,
         turns=[
            "Looking for my first credit card as a student.",
            "I'm 22, studying full-time, no income to speak of.",
         ], expected_offer_id='STUDENT_STARTER', expected_path='proceed'),

    # ---- cashback / generic ----
    dict(split='cashback', customer_id=501,
         turns=[
            "I just want a simple card with rewards.",
            "I earn about $65K and spend on everyday stuff — groceries, gas.",
         ], expected_offer_id='CASHBACK_EVERYDAY', expected_path='proceed'),
    dict(split='cashback', customer_id=502,
         turns=[
            "Nothing fancy, just a card with cash back.",
            "I'm 40, employed, income $50K.",
         ], expected_offer_id='CASHBACK_EVERYDAY', expected_path='proceed'),

    # ---- risk decline ----
    # HIGH-risk customer profiles: the guardrail should fire off the DB row
    # (risk_tier == HIGH, or debt-to-income > 0.6, or unemployed with debt).
    dict(split='risk-decline', customer_id=901,
         turns=[
            "I'd like to apply for a new credit card.",
            "I'm 52, unemployed at the moment, and I already have about $18,000 in credit card debt. Annual income is basically zero right now.",
         ], expected_offer_id=None, expected_path='decline'),
    dict(split='risk-decline', customer_id=902,
         turns=[
            "Looking for a card, ideally something with a high limit.",
            "I make around $35K a year but I already owe close to $30K in existing debt across a few cards — my credit tier is HIGH risk.",
         ], expected_offer_id=None, expected_path='decline'),
]

print(f'{len(EXAMPLES)} examples ready to upload.')

## Create the dataset (idempotent)

In [ ]:
existing = list(client.list_datasets(dataset_name=DATASET_NAME))
if existing:
    dataset = existing[0]
    print(f'Reusing existing dataset {dataset.id}')
else:
    dataset = client.create_dataset(
        dataset_name=DATASET_NAME,
        description='Scripted multi-turn customer conversations for the NBA credit-card chatbot demo.',
    )
    print(f'Created dataset {dataset.id}')

In [ ]:
# Idempotent upload: delete any existing examples on the dataset first, then
# re-create them from EXAMPLES.  Safe to re-run whenever you edit the list
# above.
existing_examples = list(client.list_examples(dataset_id=dataset.id))
if existing_examples:
    ids = [e.id for e in existing_examples]
    client.delete_examples(example_ids=ids)
    print(f'Deleted {len(ids)} stale examples.')

# We upload as individual examples so we can attach a `split` per example.
for ex in EXAMPLES:
    inputs = {'turns': ex['turns'], 'customer_id': ex['customer_id']}
    outputs = {
        'expected_offer_id': ex['expected_offer_id'],
        'expected_path': ex['expected_path'],
        'min_turns_to_offer': 2,
        'max_turns_to_offer': 4,
    }
    client.create_example(
        inputs=inputs,
        outputs=outputs,
        dataset_id=dataset.id,
        split=ex['split'],
    )
print(f'Uploaded {len(EXAMPLES)} examples.')

## Verify

In [ ]:
examples = list(client.list_examples(dataset_name=DATASET_NAME))
print(f'Total examples in `{DATASET_NAME}`: {len(examples)}')
for e in examples[:3]:
    print(e.inputs.get('turns'), '->', e.outputs.get('expected_offer_id'))